# 08 — Phase 1.A: hand-correct the desktop test split (Roboflow)

**Goal of this notebook (Phase 1.A.1 + 1.A.2):** turn the auto-labelled test split — which the headline desktop fine-tune was *evaluated against* — into a hand-corrected ground-truth split. Without this, the 0.7176 mAP@.5 in §4.1 measures *agreement with the auto-labeller*, not absolute correctness, and is not directly comparable to the source baseline's 0.4505 (real human GT). After this notebook the comparison is honest.

## What we do

1. **Bootstrap** the test split images + auto-labels onto a fresh Colab `/content/desktop_finetune/` (or use what's already there).
2. **Preview** the current auto-labels with Ultralytics overlays so you can see what needs fixing (e.g. the Windows 11 Save button missed in the 5 May 2026 demo).
3. **Package** the test split into a single zip you upload to Roboflow.
4. *(You hand-correct in Roboflow's web UI — instructions in Section 3.)*
5. **Import** the corrected zip back into Drive, replacing the old test labels.
6. **Verify** (Phase 1.A.2): count per-class instances, sanity-check, write `tables/desktop_test_handcorrected.csv` for the report.

## Why Roboflow and not LabelImg / CVAT

- 7–8 images is too small to justify standing up CVAT.
- Roboflow free-tier accepts the zip directly, imports auto-labels as starting annotations (you only correct, you don't re-label from scratch), and exports YOLOv8 labels back as a single zip.
- LabelImg works fine but requires a local Python install and one-image-at-a-time. Roboflow's grid view is faster for 8 images.
- If you prefer LabelImg or CVAT: the zip in Section 2 is YOLO-format and works in any tool that imports YOLO.

## Roadmap pointer

Tracks Plan §L.1.A in `docs/VisClick_Detailed_Plan.md`. After this notebook, mark `1.A.1` and `1.A.2` complete and move on to `08_phase1B_ablations.ipynb`.

## 0 — Setup (mount Drive, restore test split if needed)

Idempotent: if `/content/desktop_finetune/{images,labels}/test/` already has the 7–8 test items (e.g. you just ran `06`), this cell is a no-op. Otherwise it extracts `<DRIVE>/data/desktop_finetune_bundles/test.tar.gz` (the persistent bundle written by `06.3`).

In [ ]:
import os, glob, tarfile, shutil, time, json
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive/visclick"
except ImportError:
    DRIVE = os.environ.get("VISCLICK_DRIVE", "./drive")
    Path(DRIVE).mkdir(parents=True, exist_ok=True)

ROOT      = "/content/desktop_finetune"
BUNDLES   = os.path.join(DRIVE, "data", "desktop_finetune_bundles")
TEST_IMGS = os.path.join(ROOT, "images", "test")
TEST_LBLS = os.path.join(ROOT, "labels", "test")
Path(TEST_IMGS).mkdir(parents=True, exist_ok=True)
Path(TEST_LBLS).mkdir(parents=True, exist_ok=True)

def _count(p, exts):
    return len([f for f in os.listdir(p) if f.lower().endswith(exts)]) if os.path.isdir(p) else 0

n_imgs = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
n_lbls = _count(TEST_LBLS, (".txt",))
if n_imgs >= 5 and n_lbls >= 5:
    print(f"[skip] test split already present: {n_imgs} images / {n_lbls} labels")
else:
    bundle = os.path.join(BUNDLES, "test.tar.gz")
    assert os.path.isfile(bundle), (
        f"missing {bundle}.\n"
        f"Run 06_finetune_desktop.ipynb up to 6.3 first to produce the test bundle."
    )
    print(f"[restore] extracting {bundle}")
    with tarfile.open(bundle, "r:gz") as tf:
        tf.extractall(ROOT)
    n_imgs = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
    n_lbls = _count(TEST_LBLS, (".txt",))
    print(f"[restore] now {n_imgs} images / {n_lbls} labels")

assert n_imgs == n_lbls, f"mismatch: {n_imgs} images vs {n_lbls} labels — fix before continuing"
print(f"REPORT step = SETUP | n_test = {n_imgs}")

## 1 — Preview current auto-labels (so you know what to fix)

Renders the 7–8 test images with their auto-label overlays. Look for:
- Save / OK / Apply buttons that the auto-labeller missed (the live demo's classic failure case).
- `text` boxes that should actually be `button`, `text_input`, or `menu`.
- Dropdown arrows / close-X / settings cogs not labelled at all (these are `icon`).
- Wrong class assignments overall.

These are the corrections you'll make in Roboflow.

In [ ]:
import matplotlib.pyplot as plt
import cv2

CLS = ["button", "text", "text_input", "icon", "menu", "checkbox"]
COLOURS = [(255,80,80),(255,200,0),(0,180,255),(40,200,40),(180,80,255),(255,160,0)]

def draw_yolo(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    if not os.path.isfile(lbl_path):
        return img, 0
    n = 0
    with open(lbl_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                continue
            c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            x1 = int((xc - w/2) * W); y1 = int((yc - h/2) * H)
            x2 = int((xc + w/2) * W); y2 = int((yc + h/2) * H)
            colour = COLOURS[c % len(COLOURS)]
            cv2.rectangle(img, (x1,y1), (x2,y2), colour, 3)
            cv2.putText(img, CLS[c], (x1, max(y1-5, 12)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, colour, 2)
            n += 1
    return img, n

imgs = sorted(glob.glob(os.path.join(TEST_IMGS, "*")))
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, ip in zip(axes.flat, imgs[:8]):
    stem = os.path.splitext(os.path.basename(ip))[0]
    lp = os.path.join(TEST_LBLS, stem + ".txt")
    drawn, nb = draw_yolo(ip, lp)
    ax.imshow(drawn); ax.axis("off")
    ax.set_title(f"{os.path.basename(ip)}  ({nb} boxes)", fontsize=10)
for ax in axes.flat[len(imgs):]:
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Above are the test images with their CURRENT (auto-labelled) annotations.")
print("In the next step we package these for Roboflow upload; you correct them there.")

## 2 — Package test split into a Roboflow-ready zip

Produces a single `desktop_test_for_handlabel.zip` containing:
- `images/` — the 7–8 test PNG/JPG files
- `labels/` — the 7–8 YOLO `.txt` files (auto-labels, to be corrected)
- `data.yaml` — Roboflow reads this to set up the 6-class schema, so when you upload, the classes are already `button / text / text_input / icon / menu / checkbox`.

The zip lands at `<DRIVE>/data/desktop_test_for_handlabel.zip` — open Drive in your browser, right-click → Download. **Section 3 below has the step-by-step Roboflow workflow.**

In [ ]:
import zipfile

DATA_YAML = (
    "# generated by 08_phase1A_handlabel.ipynb\n"
    "path: .\n"
    "train: images\n"
    "val: images\n"
    "test: images\n"
    "\n"
    "nc: 6\n"
    "names: ['button', 'text', 'text_input', 'icon', 'menu', 'checkbox']\n"
)

OUT_ZIP_DIR = os.path.join(DRIVE, "data")
Path(OUT_ZIP_DIR).mkdir(parents=True, exist_ok=True)
OUT_ZIP = os.path.join(OUT_ZIP_DIR, "desktop_test_for_handlabel.zip")

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for ip in sorted(glob.glob(os.path.join(TEST_IMGS, "*"))):
        zf.write(ip, arcname=os.path.join("images", os.path.basename(ip)))
    for lp in sorted(glob.glob(os.path.join(TEST_LBLS, "*.txt"))):
        zf.write(lp, arcname=os.path.join("labels", os.path.basename(lp)))
    zf.writestr("data.yaml", DATA_YAML)

size_mb = os.path.getsize(OUT_ZIP) / 1024 / 1024
print(f"REPORT step = PACKAGE | path = {OUT_ZIP} | size_mb = {size_mb:.2f}")
print()
print("Next step:")
print(f"  1. Open Drive in your browser at {OUT_ZIP}")
print(f"  2. Right-click → Download (zip is ~{size_mb:.1f} MB)")
print(f"  3. Follow Section 3 below to upload + correct in Roboflow.")

## 3 — Roboflow workflow (the manual part), step by step

Two paths, same result. Pick whichever fits your time budget.

| Path | Tool | Time for 8 images | Quality |
|------|------|-------------------|---------|
| **A.** Auto Label with LLM prompt + review | Roboflow's built-in VLM (uses our class prompt) | ~2 min auto + ~10 min review = **~12 min** | Good after review |
| **B.** Manual from auto-labels (no LLM) | Roboflow annotation tool | **~30 min** | Reliable |

I recommend **Path A**. The free tier gives enough Auto Label credits to do 8 images several times over.

---

### 3.1 — Sign up + create the project (5 min, one-time)

1. Open <https://app.roboflow.com/> in your browser.
2. **Sign up** with Google or email (free tier is fine).
3. After signup you land in your workspace. If a workspace named `visclick` doesn't exist, click the workspace dropdown (top-left) → **Create New Workspace** → name it `visclick`. Select **Public Plan – Free** (this is fine; only YOU see private projects).
4. In your workspace, click the big **+ New Project** card.
5. Project setup form:
   - **Project Name:** `visclick-desktop-test-handlabel`
   - **License:** `Private` (yes, on the free tier you still get *Private*-by-default for projects with ≤ 10k images)
   - **Project Type:** `Object Detection`
   - **What will your model predict?** type `ui_element` (this is the "annotation group" name; it doesn't affect class labels).
6. Click **Create Public Project** (the button name is misleading — your data stays private; only the *project metadata* is public on the free tier, which is fine for our case).

You now have an empty project page.

---

### 3.2 — Upload the zip (2 min)

1. On the project's home page, click **Upload Data** (left sidebar) → **Upload**.
2. Drag `desktop_test_for_handlabel.zip` (the one you downloaded from Drive in Section 2) onto the upload area, OR click **Select Files / Folder** and pick the zip.
3. Roboflow shows the contents: `images/` (7–8 PNG/JPG) + `labels/` (7–8 YOLO `.txt`) + `data.yaml`.
4. It will say *"YOLO labels detected"* and show the 6 classes (`button / text / text_input / icon / menu / checkbox`). **Confirm classes match.** If Roboflow renames them (e.g. dashes), click **Edit** and rename back to exactly those six.
5. Click **Save and Continue**.
6. **Assign images to a split** dialog: pick **Add to → Annotate** (so all 8 land in the unannotated queue). Click **Save and Continue**.

You should now see all 8 images on the project's *Annotate* tab, each showing the auto-label boxes already drawn.

---

### 3.3 — Path A: Auto Label with LLM prompt (~12 min total)

#### 3.3.a — Open the Auto Label feature

1. On the *Annotate* tab, click any one image to open Roboflow's annotation tool.
2. In the right-side panel look for **Label Assist** (sometimes called **Auto Label** or **Smart Label** depending on Roboflow's version). It's the magic-wand icon, usually near the top of the right sidebar.
3. Click it → choose **"Use a foundation model" / "Auto Label with prompt"** (exact wording varies; you want the option that takes a TEXT PROMPT, not the one that uses your own pre-trained model).

If you don't see Auto Label / Label Assist:
- Older free accounts: it might be under **Workflows** instead. Click **Workflows** in the left sidebar → **Auto Label** template. Same idea.
- Brand new free accounts: occasionally you have to click *"Try Auto Label"* on the project home and confirm credits. Confirm; the 8 images cost a tiny fraction of monthly credits.

#### 3.3.b — Paste this exact prompt

Copy the block below as-is and paste into the Auto Label prompt box:

```
You are labelling clickable user-interface elements in a desktop screenshot for an automation tool.

For every element that is interactive or important to read, draw a tight bounding box and assign EXACTLY ONE of these six classes:

1. button — Rectangular clickable controls with a text label or text+icon, usually 40–250 px wide. Examples: Save, Cancel, OK, Apply, Submit, Send, Close (when shaped as a button), Open. Include both filled and outlined Windows-11 / Material-style buttons. Tab-style controls also count as button.

2. text — Static text labels, headings, captions, paragraphs, file sizes, date stamps. NOT clickable (or only marginally so). Do NOT use 'text' for content that is clearly a button or a list/menu entry — use those classes instead.

3. text_input — Editable text fields the user types into. Examples: file name field, search bar, URL/address bar, breadcrumb path bar that accepts typing. Usually has a visible border and may contain placeholder text.

4. icon — Small image-only clickable, typically 16–40 px square, NO visible text inside the box. Examples: close-X in the title bar, settings/gear cog, hamburger ≡, dropdown chevron (▼ or ^v) when standalone, back/forward/up navigation arrows, app icons in the taskbar, sidebar 'Home' / 'Gallery' icons (the icon part only, not the label).

5. menu — Items inside a list, navigation drawer, dropdown, menu bar, or sidebar list. Examples: each sidebar entry in File Explorer (Documents, Downloads, Desktop) — box the entire row including its tiny icon and the label text; each file row in a file list; each item in 'Save as type' dropdown; breadcrumb segments that show a folder name; ribbon items.

6. checkbox — Square checkboxes (checked or unchecked) and round radio buttons. Small, ~14–22 px.

Important rules:
- One element = one box. Do not draw overlapping boxes for the same control.
- Box should hug the element with at most ~3 px slack.
- Skip purely decorative graphics, app logos that aren't clickable, and image content (photos, file thumbnails, video frames).
- If unsure between 'button' and 'menu', prefer 'menu' for entries that live inside a list/sidebar, prefer 'button' for standalone controls.
- If unsure between 'icon' and 'button', prefer 'icon' if the element is < 40 px in either dimension and contains no text; prefer 'button' otherwise.
- Save / OK / Apply / Cancel are ALWAYS class 'button', even on flat Windows 11 dialogs where they look subtle.
```

4. Click **Run Auto Label** (or the equivalent "Generate" button).
5. Wait ~30 seconds per image. Roboflow runs the VLM and writes new boxes onto each image.

#### 3.3.c — Review every image (this is the important part)

VLMs are *good* but not infallible. Open each of the 8 images in turn and do a quick pass:

- ✅ **Save / OK / Apply / Cancel buttons present and labelled `button`?** This is the most common LLM miss; add it manually if not.
- ✅ **Each sidebar entry labelled `menu`?** The LLM sometimes labels them as `text`. Re-label.
- ✅ **Close-X / settings cog / dropdown arrows labelled `icon`?** LLM frequently misses these because they're small.
- ✅ **No duplicate boxes on the same element?** If yes, delete the smaller / less-tight one.
- ✅ **Address bar / file-name field labelled `text_input`?** Usually the LLM gets this right.

For each fix:
- **Add a missing box:** press `B` then drag from top-left to bottom-right corner of the element. Pick its class from the right sidebar.
- **Change a class:** click the box → in the right sidebar, change the class in the dropdown.
- **Delete a box:** click it → press `Del` (or `D` depending on Roboflow build).
- **Move to next image:** press `Tab` (or arrow keys / use the image grid in the left sidebar).

Most images take ~1 minute of review. If a single image is taking > 3 minutes, skip the most-trivial fixes — perfection isn't required.

---

### 3.4 — Path B: Manual annotation from auto-labels (~30 min total)

Use this if Auto Label isn't available or its credits are exhausted. The auto-labels imported from your zip are already on each image; you just correct them.

For each of the 8 images, open it in the annotation tool and:

**(a) Add missed clickables.** Compare against the live screenshot. Press `B` → drag a box around the missed element → pick class from sidebar. The Save / OK / Apply buttons are the most common misses on flat Windows 11 dialogs.

**(b) Fix wrong class labels.** Click the box → change class via the right-sidebar dropdown.

**(c) Add icon-only clickables.** Close-X title-bar buttons, settings cogs, dropdown arrows (`▼` / `^v`), minimise / maximise, app-bar icons. Class = `icon`.

**(d) Delete spurious boxes.** Hallucinated by the auto-labeller. Click → `Del`.

**(e) Tighten loose boxes.** Drag corners until the box hugs the element. ~3 px slack is fine.

Useful keys (Roboflow web): `B` draw box · `Del` delete selected · `1`–`6` directly assign that class to a selected box · `Tab` next image · `Shift+Tab` previous image · `S` save image · `Esc` deselect.

---

### 3.5 — Export the corrected dataset (3 min)

1. Left sidebar → **Versions** → **Generate New Version**.
2. **Preprocessing:** leave defaults (Auto-Orient ON, Resize OFF — we don't want Roboflow to resize our 1920×1080 screenshots).
3. **Augmentation:** click *Skip* / *No augmentation*. We're exporting test labels, not training data; augmentation is forbidden.
4. Click **Generate** → wait ~30 seconds.
5. The new version page → **Export Dataset** button (or top-right *Download Dataset*).
6. **Format:** `YOLOv8` (this gives the right `data.yaml` + folder structure).
7. **Download:** select **download zip to computer** (NOT *show download code*).
8. The zip lands in your Downloads folder. Filename will be like `visclick-desktop-test-handlabel.v1i.yolov8.zip`.

---

### 3.6 — Upload back to Drive (2 min)

The notebook's Section 4 expects the corrected zip at exactly this path:
```
<DRIVE>/data/desktop_test_handcorrected.zip
```

To get it there:
1. Rename the downloaded file to `desktop_test_handcorrected.zip`.
2. Open <https://drive.google.com/> in your browser → navigate to `My Drive / visclick / data /`.
3. Drag the renamed zip onto the page → wait for the green tick.

---

### 3.7 — Quick sanity check before running Section 4

Open the zip locally (Windows Explorer can preview zip contents) and confirm:
- A folder like `train/images/` (or `images/`) with 7–8 PNG/JPG files.
- A matching folder `train/labels/` (or `labels/`) with 7–8 `.txt` files.
- A `data.yaml` at the zip root with `nc: 6` and the names list.

If Roboflow renamed the classes (sometimes it strips underscores → `text input` instead of `text_input`) **don't fix it manually** — Section 4's import code reads `data.yaml` and remaps automatically, including renamed classes. The only condition is that the names are still recognisable as the original six.

When ready, return to Colab and run Section 4 (cell 9) and Section 5 (cell 11). Then commit `Phase 1.A done`.

## 4 — Import corrected labels back into the test split

Set `ROBOFLOW_ZIP` to the path of the zip you downloaded from Roboflow and uploaded to Drive. Default path matches the suggestion in Section 3.5.

What this cell does:
1. Extract the Roboflow zip into a temp dir.
2. Read its `data.yaml` to determine how Roboflow ordered the classes.
3. Remap each label file's class indices into VisClick's canonical order (`button=0 / text=1 / text_input=2 / icon=3 / menu=4 / checkbox=5`).
4. Replace `<DRIVE>/data/desktop_finetune_bundles/test.tar.gz` with a corrected bundle (so future runs of `06`, `08_phase1B_*` use the corrected labels automatically).
5. Also write a backup `test_autolabels.tar.gz` of the original bundle so you can revert if needed.

In [ ]:
import yaml

ROBOFLOW_ZIP = os.path.join(DRIVE, "data", "desktop_test_handcorrected.zip")
assert os.path.isfile(ROBOFLOW_ZIP), (
    f"Missing {ROBOFLOW_ZIP}.\n"
    f"Upload the YOLOv8 zip you downloaded from Roboflow to that path on Drive."
)

TMP = "/content/_rf_tmp"
shutil.rmtree(TMP, ignore_errors=True); Path(TMP).mkdir(parents=True)
with zipfile.ZipFile(ROBOFLOW_ZIP, "r") as zf:
    zf.extractall(TMP)

yml_path = None
for root, _, files in os.walk(TMP):
    if "data.yaml" in files:
        yml_path = os.path.join(root, "data.yaml"); break
assert yml_path, "data.yaml missing from Roboflow zip"
with open(yml_path) as f:
    rf_yaml = yaml.safe_load(f)
rf_names = list(rf_yaml.get("names", []))
print(f"Roboflow class order: {rf_names}")

VC_NAMES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

def _normalise(s):
    return (s or "").strip().lower().replace("-", "_").replace(" ", "_")

VC_NORM = [_normalise(n) for n in VC_NAMES]
ALIASES = {
    "buttons": "button", "btn": "button", "clickable": "button",
    "label": "text", "static_text": "text", "caption": "text",
    "input": "text_input", "textfield": "text_input", "text_field": "text_input",
    "editbox": "text_input", "edit_text": "text_input",
    "image": "icon", "image_button": "icon", "icon_button": "icon",
    "close": "icon", "chevron": "icon", "arrow": "icon",
    "menu_item": "menu", "list_item": "menu", "nav": "menu",
    "navigation": "menu", "sidebar_item": "menu", "dropdown": "menu",
    "radio": "checkbox", "radio_button": "checkbox", "toggle": "checkbox",
}

rf_to_vc = {}
for i, n in enumerate(rf_names):
    norm = _normalise(n)
    target = None
    if norm in VC_NORM:
        target = VC_NAMES[VC_NORM.index(norm)]
    elif norm in ALIASES:
        target = ALIASES[norm]
    if target:
        rf_to_vc[i] = VC_NAMES.index(target)
        if target != n:
            print(f"  remap: Roboflow class {n!r} -> VisClick {target!r}")
    else:
        print(f"  WARNING: Roboflow class {n!r} not recognised; will drop those boxes")

lbl_dirs = []
for root, _, files in os.walk(TMP):
    if any(f.endswith(".txt") for f in files):
        if root.endswith("labels") or os.path.basename(os.path.dirname(root)) == "labels":
            lbl_dirs.append(root)
img_dirs = []
for root, _, files in os.walk(TMP):
    if any(f.lower().endswith((".png", ".jpg", ".jpeg")) for f in files):
        if root.endswith("images") or os.path.basename(os.path.dirname(root)) == "images":
            img_dirs.append(root)
print(f"label dirs found: {lbl_dirs}\nimage dirs found: {img_dirs}")

if os.path.isfile(os.path.join(BUNDLES, "test.tar.gz")):
    backup = os.path.join(BUNDLES, "test_autolabels.tar.gz")
    if not os.path.isfile(backup):
        shutil.copy2(os.path.join(BUNDLES, "test.tar.gz"), backup)
        print(f"[backup] saved auto-label bundle to {backup}")

shutil.rmtree(TEST_IMGS, ignore_errors=True); Path(TEST_IMGS).mkdir(parents=True)
shutil.rmtree(TEST_LBLS, ignore_errors=True); Path(TEST_LBLS).mkdir(parents=True)

for d in img_dirs:
    for fn in os.listdir(d):
        if fn.lower().endswith((".png", ".jpg", ".jpeg")):
            shutil.copy2(os.path.join(d, fn), os.path.join(TEST_IMGS, fn))
for d in lbl_dirs:
    for fn in os.listdir(d):
        if not fn.endswith(".txt"):
            continue
        out = os.path.join(TEST_LBLS, fn)
        kept = 0
        with open(os.path.join(d, fn)) as fin, open(out, "w") as fout:
            for line in fin:
                parts = line.split()
                if len(parts) != 5:
                    continue
                c = int(parts[0])
                if c not in rf_to_vc:
                    continue
                fout.write(" ".join([str(rf_to_vc[c])] + parts[1:]) + "\n")
                kept += 1
        print(f"  {fn}: kept {kept} boxes")

n_imgs_new = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
n_lbls_new = _count(TEST_LBLS, (".txt",))
print(f"REPORT step = IMPORT | n_imgs = {n_imgs_new} | n_labels = {n_lbls_new}")

import tarfile as _tar
new_bundle = os.path.join(BUNDLES, "test.tar.gz")
with _tar.open(new_bundle, "w:gz") as tf:
    for sub in ("images/test", "labels/test"):
        full = os.path.join(ROOT, sub)
        for fn in sorted(os.listdir(full)):
            tf.add(os.path.join(full, fn), arcname=os.path.join(sub, fn))
print(f"REPORT step = REPACK | new test.tar.gz = {os.path.getsize(new_bundle)} bytes")

## 5 — Phase 1.A.2: verify and write report row

Counts per-class instances on the corrected test split, sanity-checks (no negative coords, no out-of-range classes), and writes `<DRIVE>/reports/tables/desktop_test_handcorrected.csv` for direct paste into Report §4.6b.

In [ ]:
from collections import Counter
import csv

VC_NAMES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
counts = Counter()
errors = []
n_imgs = 0; n_boxes = 0

for ip in sorted(glob.glob(os.path.join(TEST_IMGS, "*"))):
    stem = os.path.splitext(os.path.basename(ip))[0]
    lp = os.path.join(TEST_LBLS, stem + ".txt")
    if not os.path.isfile(lp):
        errors.append(f"missing label for {os.path.basename(ip)}"); continue
    n_imgs += 1
    with open(lp) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                errors.append(f"{stem}: malformed line: {line.strip()!r}"); continue
            c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            if not (0 <= c < 6):
                errors.append(f"{stem}: class out of range: {c}"); continue
            for v, n in zip([xc, yc, w, h], ["xc", "yc", "w", "h"]):
                if not (0 <= v <= 1):
                    errors.append(f"{stem}: {n}={v} not in [0,1]")
            counts[VC_NAMES[c]] += 1
            n_boxes += 1

print(f"\nREPORT §4.6b — hand-corrected test split:")
print(f"  images : {n_imgs}")
print(f"  boxes  : {n_boxes}")
for cls in VC_NAMES:
    print(f"    {cls:11s} : {counts.get(cls, 0)}")
if errors:
    print("\nWARNINGS:")
    for e in errors[:10]:
        print("  " + e)
    if len(errors) > 10:
        print(f"  ... ({len(errors) - 10} more)")
else:
    print("\n✓ no warnings — labels look clean")

TABLES = os.path.join(DRIVE, "reports", "tables")
Path(TABLES).mkdir(parents=True, exist_ok=True)
csv_path = os.path.join(TABLES, "desktop_test_handcorrected.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["split", "images", "total_boxes"] + VC_NAMES + ["date"])
    w.writerow(["test", n_imgs, n_boxes] + [counts.get(c, 0) for c in VC_NAMES] +
                [time.strftime("%Y-%m-%d")])
print(f"\nREPORT step = VERIFY | path = {csv_path}")
print("Paste these counts into Report §4.6b. Then mark Plan 1.A.1 and 1.A.2 done and run 08_phase1B_ablations.ipynb.")

## 7 — Sync `desktop_test_handcorrected.csv` into the GitHub repo

So the per-class counts can be cited from a permanent github URL, run the sync helper. Then commit + push manually with your token.

In [ ]:
import os, subprocess

REPO = "/content/visclick"
if not os.path.isdir(REPO):
    print("cloning repo first…")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/HiranMadhu/visclick.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)

subprocess.run(
    ["git", "-C", REPO, "config", "user.email", "phase1a@colab.local"], check=False)
subprocess.run(
    ["git", "-C", REPO, "config", "user.name",  "VisClick Colab Bot"],   check=False)

subprocess.run(["python", f"{REPO}/scripts/sync_reports_to_repo.py",
                "--drive", DRIVE, "--repo", REPO], check=True)

r = subprocess.run(["git", "-C", REPO, "status", "--porcelain", "reports/"],
                   capture_output=True, text=True, check=True)
if not r.stdout.strip():
    print("\nNothing to commit under reports/ — already up to date.")
else:
    subprocess.run(["git", "-C", REPO, "add", "reports/"], check=True)
    cm = subprocess.run(
        ["git", "-C", REPO, "commit", "-m", "reports: phase1A handcorrected counts"],
        capture_output=True, text=True)
    print(cm.stdout.strip()); print(cm.stderr.strip())

rb = subprocess.run(
    ["git", "-C", REPO, "pull", "--rebase", "origin", "main"],
    capture_output=True, text=True)
print(rb.stdout.strip()); print(rb.stderr.strip())

print("\nNow run THIS line in a new cell, with your token pasted in:")
print()
print(f"!cd {REPO} && git push https://HiranMadhu:<PASTE_TOKEN>@github.com/HiranMadhu/visclick.git HEAD:main")

## 6 — What's next

- [x] **1.A.1** Hand-correct the 8 desktop test images in Roboflow → done above.
- [x] **1.A.2** Verify exports → CSV written to `<DRIVE>/reports/tables/desktop_test_handcorrected.csv`.
- [ ] **1.B.1–5** Run `08_phase1B_ablations.ipynb` (will be added next): trains M0 / M1 / M2, re-evaluates M3 (= current desktop FT) on this hand-corrected test set, dumps `tables/transfer_experiments.csv`.
- [ ] **1.C.1–3** On your Windows machine, run the three baseline scripts (`scripts/baseline_template.py` / `baseline_ocr_only.py` / `baseline_pywinauto.py`) — these will be added in a separate commit.

Update Plan §L.1 checkboxes and commit with a message like:

```
Phase 1.A.1: hand-corrected 8 desktop test images (n_boxes=NN)
Phase 1.A.2: verified counts, wrote tables/desktop_test_handcorrected.csv
```